In [2]:
import sys
#!{sys.executable} -m pip install ultralytics opencv-python pandas matplotlib tqdm
import os
import cv2
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO, RTDETR

print("Бібліотеки успішно завантажено!")

Бібліотеки успішно завантажено!


In [3]:
raw_dir = 'URPC2020'
enh_dir = 'URPC2020_Enhanced'

In [10]:
from tqdm import tqdm

def white_balance(img):
    # Алгоритм Grey World для балансу білого
    result = img.copy().astype(np.float32)
    avg_b = np.average(result[:, :, 0])
    avg_g = np.average(result[:, :, 1])
    avg_r = np.average(result[:, :, 2])
    
    avg = (avg_b + avg_g + avg_r) / 3
    if avg_b > 0: result[:, :, 0] = np.clip(result[:, :, 0] * (avg / avg_b), 0, 255)
    if avg_g > 0: result[:, :, 1] = np.clip(result[:, :, 1] * (avg / avg_g), 0, 255)
    if avg_r > 0: result[:, :, 2] = np.clip(result[:, :, 2] * (avg / avg_r), 0, 255)
    
    return result.astype(np.uint8)

def adjust_gamma(image, gamma=0.7):
    # Гамма-корекція
    invGamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    return cv2.LUT(image, table)

def process_and_copy_dataset():
    splits = ['train', 'valid', 'test']
    
    for split in splits:
        img_src_dir = os.path.join(raw_dir, split, 'images')
        lbl_src_dir = os.path.join(raw_dir, split, 'labels')
        
        img_dst_dir = os.path.join(enh_dir, split, 'images')
        lbl_dst_dir = os.path.join(enh_dir, split, 'labels')
        
        os.makedirs(img_dst_dir, exist_ok=True)
        
        # Копіюємо мітки без змін
        if os.path.exists(lbl_src_dir):
            shutil.copytree(lbl_src_dir, lbl_dst_dir, dirs_exist_ok=True)
        
        if not os.path.exists(img_src_dir):
            continue
            
        images = [f for f in os.listdir(img_src_dir) if f.endswith(('.jpg', '.png'))]
        
        for img_name in tqdm(images, desc=f'Покращення {split}'):
            img_path = os.path.join(img_src_dir, img_name)
            out_path = os.path.join(img_dst_dir, img_name)
            
            img = cv2.imread(img_path)
            if img is None:
                continue
                
            # Застосовуємо методи з роботи: WB -> Gamma
            img_wb = white_balance(img)
            img_final = adjust_gamma(img_wb, gamma=0.7)
            
            cv2.imwrite(out_path, img_final)

print("Розпочато генерацію покращеного датасету")
process_and_copy_dataset()
print("Покращений датасет успішно створено у папці URPC2020_Enhanced!")

Розпочато генерацію покращеного датасету...



Покращення train: 100%|████████████████████████████████████████████████████████████| 5543/5543 [12:26<00:00,  7.42it/s]

Покращення valid: 100%|████████████████████████████████████████████████████████████| 1200/1200 [05:20<00:00,  3.74it/s]

Покращення test: 100%|███████████████████████████████████████████████████████████████| 800/800 [03:36<00:00,  3.70it/s]

Покращений датасет успішно створено у папці URPC2020_Enhanced!


In [4]:
raw_yaml = f"""
path: {os.path.abspath('URPC2020')}
train: train/images
val: valid/images
test: test/images

names:
  0: holothurian
  1: echinus
  2: scallop
  3: starfish
"""

enh_yaml = f"""
path: {os.path.abspath('URPC2020_Enhanced')}
train: train/images
val: valid/images
test: test/images

names:
  0: holothurian
  1: echinus
  2: scallop
  3: starfish
"""

with open('urpc_raw.yaml', 'w') as f:
    f.write(raw_yaml)
    
with open('urpc_enhanced.yaml', 'w') as f:
    f.write(enh_yaml)

print("Файли конфігурації urpc_raw.yaml та urpc_enhanced.yaml створено")

Файли конфігурації urpc_raw.yaml та urpc_enhanced.yaml створено


In [ ]:
import gc
import torch

epochs = 50
imgsz = 512

print("=== 1. Навчання YOLOv8m (raw data) ===")
model_yolo_raw = YOLO('yolov8m.pt')
model_yolo_raw.train(data='urpc_raw.yaml', epochs=epochs, imgsz=imgsz, batch=4, project='UOD_Results', name='yolo_raw', device=0)

# Очищення пам'яті
del model_yolo_raw
gc.collect()
torch.cuda.empty_cache()

print("\n=== 2. Навчання YOLOv8m (Покращені дані: WB + Gamma) ===")
model_yolo_enh = YOLO('yolov8m.pt')
model_yolo_enh.train(data='urpc_enhanced.yaml', epochs=epochs, imgsz=imgsz, batch=4, project='UOD_Results', name='yolo_enhanced', device=0)

del model_yolo_enh
gc.collect()
torch.cuda.empty_cache()



In [10]:
epochs = 20
imgsz = 512

print("\n=== 3. Навчання RT-DETR-l (raw data) ===")
model_rtdetr_raw = RTDETR('rtdetr-l.pt')
model_rtdetr_raw.train(data='urpc_raw.yaml', epochs=epochs, imgsz=imgsz, batch=1, project='UOD_Results', name='rtdetr_raw', device=0, deterministic=False)

del model_rtdetr_raw
gc.collect()
torch.cuda.empty_cache()

print("\n=== 4. Навчання RT-DETR-l (Покращені дані: WB + Gamma) ===")
model_rtdetr_enh = RTDETR('rtdetr-l.pt')
model_rtdetr_enh.train(data='urpc_enhanced.yaml', epochs=epochs, imgsz=imgsz, batch=1, project='UOD_Results', name='rtdetr_enhanced', device=0, deterministic=False)

del model_rtdetr_enh
gc.collect()
torch.cuda.empty_cache()

print("Етапи навчання завершено. Ваги збережено у папці UOD_Results.")


=== 3. Навчання RT-DETR-l (raw data) ===
Ultralytics 8.4.53  Python-3.8.10 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=urpc_raw.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=rtdetr_raw-2, nbs=64, nms=False, opset=None, optimize

In [ ]:
!{sys.executable} -m pip install --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [1]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.4.1+cu121
12.1
True
NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [13]:
def evaluate_model(weights_path, yaml_data, model_type, preprocessing_name):
    print(f"Збір метрик для {model_type} ({preprocessing_name})...")
    
    if 'RT-DETR' in model_type:
        model = RTDETR(weights_path)
    else:
        model = YOLO(weights_path)
        
    # Запуск тестування на папці test
    metrics = model.val(data=yaml_data, split='test', verbose=False)
    
    # Обчислення швидкості на кадр
    preprocess = metrics.speed['preprocess']
    inference = metrics.speed['inference']
    postprocess = metrics.speed['postprocess']
    total_time_ms = preprocess + inference + postprocess
    fps = 1000.0 / total_time_ms if total_time_ms > 0 else 0
    
    return {
        'Модель': model_type,
        'Попередня обробка': preprocessing_name,
        'Precision (Точність)': round(metrics.box.mp, 4),
        'Recall (Повнота)': round(metrics.box.mr, 4),
        'mAP@0.5': round(metrics.box.map50, 4),
        'mAP@0.5:0.95': round(metrics.box.map, 4),
        'Час інференсу (мс)': round(inference, 2),
        'Швидкодія (FPS)': round(fps, 1)
    }

results = []

# Шляхи до найкращих ваг (згенеруються на попередньому етапі)
paths = [
    ('runs/detect/UOD_Results/yolo_raw/weights/best.pt', 'urpc_raw.yaml', 'YOLOv8m', 'Відсутня (Raw)'),
    ('runs/detect/UOD_Results/yolo_enhanced/weights/best.pt', 'urpc_enhanced.yaml', 'YOLOv8m', 'WB + Gamma'),
    ('runs/detect/UOD_Results/rtdetr_raw/weights/best.pt', 'urpc_raw.yaml', 'RT-DETR-l', 'Відсутня (Raw)'),
    ('runs/detect/UOD_Results/rtdetr_enhanced/weights/best.pt', 'urpc_enhanced.yaml', 'RT-DETR-l', 'WB + Gamma')
]

for weight_path, yaml_file, m_type, preproc in paths:
    if os.path.exists(weight_path):
        res = evaluate_model(weight_path, yaml_file, m_type, preproc)
        results.append(res)
    else:
        print(f"Ваги не знайдені за шляхом {weight_path}. Модель пропущено.")

# Виведення результатів
df_results = pd.DataFrame(results)
display(df_results)

Збір метрик для YOLOv8m (Відсутня (Raw))...
Ultralytics 8.4.53  Python-3.8.10 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
Model summary (fused): 93 layers, 25,842,076 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access  (ping: 0.30.1 ms, read: 412.9227.5 MB/s, size: 460.5 KB)
val: Scanning C:\Users\rex20\Documents\Thesis\URPC2020\test\labels... 800 images, 25 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 800/800 1.5Kit/s 0.5s0.0s
val: New cache created: C:\Users\rex20\Documents\Thesis\URPC2020\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 50/50 3.4it/s 14.6s0.3s
                   all        800       6581      0.812       0.68      0.764      0.432
Speed: 0.7ms preprocess, 9.7ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to C:\Users\rex20\Documents\Thesis\runs\detect\val-2
Збір метрик для YOLOv8m (WB + Gamma)...
Ultralytics 8.4.53  Python-3.8.10 torch

,Модель,Попередня обробка,Precision (Точність),Recall (Повнота),mAP@0.5,mAP@0.5:0.95,Час інференсу (мс),Швидкодія (FPS)
0,YOLOv8m,Відсутня (Raw),0.8119,0.6795,0.7638,0.4322,9.71,87.6
1,YOLOv8m,WB + Gamma,0.7948,0.6657,0.7446,0.4179,9.78,85.0
2,RT-DETR-l,Відсутня (Raw),0.7476,0.6584,0.7064,0.3810,23.95,39.3
3,RT-DETR-l,WB + Gamma,0.7631,0.6420,0.7100,0.3916,23.60,39.0
